In [ ]:
! pip install opendatasets

Downloading Dataset from kaggle :Newyork city taxi fare

In [ ]:
import opendatasets as od
dataset_url = 'https://www.kaggle.com/c/new-york-city-taxi-fare-prediction/overview'

In [ ]:
%%time
od.download(dataset_url)

100%|██████████| 1.56G/1.56G [00:18<00:00, 90.0MB/s]



Extracting archive ./new-york-city-taxi-fare-prediction/new-york-city-taxi-fare-prediction.zip to ./new-york-city-taxi-fare-prediction
CPU times: user 53.7 s, sys: 15.8 s, total: 1min 9s
Wall time: 1min 30s


In [ ]:
data_dir ='./new-york-city-taxi-fare-prediction'

In [ ]:
!ls -lh {data_dir}

ls: cannot access './new-york-city-taxi-fare-prediction': No such file or directory


In [ ]:
!wc -l {data_dir}/train.csv

wc: ./new-york-city-taxi-fare-prediction/train.csv: No such file or directory


In [ ]:
!wc -l {data_dir}/test.csv

wc: ./new-york-city-taxi-fare-prediction/test.csv: No such file or directory


In [ ]:
!wc -l {data_dir}/sample_submission.csv

wc: ./new-york-city-taxi-fare-prediction/sample_submission.csv: No such file or directory


In [ ]:
!head {data_dir}/train.csv

head: cannot open './new-york-city-taxi-fare-prediction/train.csv' for reading: No such file or directory


In [ ]:
!head {data_dir}/test.csv

head: cannot open './new-york-city-taxi-fare-prediction/test.csv' for reading: No such file or directory


DataLoading and EDA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings as wr
wr.filterwarnings('ignore')

In [ ]:
selected_cols = 'fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count'.split(',')

In [ ]:
selected_cols

['fare_amount',
 'pickup_datetime',
 'pickup_longitude',
 'pickup_latitude',
 'dropoff_longitude',
 'dropoff_latitude',
 'passenger_count']

In [ ]:
dtypes = {'fare_amount':'float32',
 'pickup_longitude':'float32',
 'pickup_latitude':'float32',
 'dropoff_longitude':'float32',
 'dropoff_latitude':'float32',
 'passenger_count':'uint8'}
import random

def skip_row(row_index):
  if row_index == 0:
    return False
  return random.random() > 0.02

In [ ]:
random.seed(42)
df = pd.read_csv(data_dir+'/train.csv',
                 usecols=selected_cols,dtype = dtypes ,parse_dates =['pickup_datetime'],
                 skiprows =skip_row)
test_df = pd.read_csv(data_dir+'/test.csv', dtype=dtypes, parse_dates=['pickup_datetime'])

FileNotFoundError: [Errno 2] No such file or directory: './new-york-city-taxi-fare-prediction/train.csv'

In [ ]:
df

In [ ]:
df['pickup_datetime'].min(),df['pickup_datetime'].max()

In [ ]:
test_df = pd.read_csv(data_dir+'/test.csv', dtype=dtypes, parse_dates=['pickup_datetime'])

In [ ]:
test_df['pickup_datetime'].min(),df['pickup_datetime'].max()

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
df.isnull().sum()

In [ ]:
df = df.dropna()

In [ ]:
df.shape

In [ ]:
df.nunique()

In [ ]:
plt.figure(figsize=(10,8))
sns.barplot(x=df['passenger_count'].value_counts().index,y=df['passenger_count'].value_counts().values)

here we can see that there are bookings with 0 passengers and more than 6 passengers. More than six passengers will not fit in seven seat taxi


In [ ]:
# since fare amount varies with distance ,location and booking date&time we extract these features


In [ ]:
def add_dateparts(df,col):
  df[col +'_year'] = df[col].dt.year
  df[col +'_month'] = df[col].dt.month
  df[col +'_day'] = df[col].dt.day
  df[col +'_hour'] = df[col].dt.hour
  df[col +'_weekday'] = df[col].dt.weekday

In [ ]:
add_dateparts(df,'pickup_datetime')
add_dateparts(test_df,'pickup_datetime')

In [ ]:
df

In [ ]:
# distance between pickup and drop location

from dataclasses import dataclass
import numpy as np

def haversine_np(lon1,lat1,lon2 ,lat2):
  """
  Calculate the great circle distance between two points
  on the earth (specified in decimal degrees)

  All args must be of equal length.

  """
  lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
  dlon = lon2-lon2
  dlat = lat2 -lat1
  a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2)* np.sin(dlon/2.0)**2
  c = 2 * np.arcsin(np.sqrt(a))
  km = 6367 * c
  return km

In [ ]:
def add_trip_distance(df):
  df['trip_distance'] = haversine_np(df['pickup_longitude'],df['pickup_latitude'],df['dropoff_longitude'],df['dropoff_latitude'])

In [ ]:
%%time
add_trip_distance(df)
add_trip_distance(test_df)

In [ ]:
df['distance_bin'] = pd.cut(df['trip_distance'], bins=[0,2,5,10,20,50])

plt.figure(figsize=(10,8))
sns.boxplot(x='distance_bin', y='fare_amount', data=df)

plt.title('Fare Distribution by Trip Distance Range')
plt.show()


In [ ]:
df.drop(columns=['distance_bin'], inplace=True)

In [ ]:
#there are so many outlier in the data

Removing Outliers

In [ ]:
def remove_outliers(df):
  return df[(df['passenger_count']>=1) &
        (df['passenger_count']< 7) &
        (df['trip_distance']> 0) &
        (df['trip_distance']< 100) &
        (df['fare_amount']> 0) &
        (df['fare_amount']< 200)]

In [ ]:
df = remove_outliers(df)

In [ ]:
df.describe().T

In [ ]:
df[df['fare_amount'] > 195]

In [ ]:
df

In [ ]:
df['pickup_datetime_year'] = df['pickup_datetime_year'].astype('category')
df['pickup_datetime_weekday'] = df['pickup_datetime_weekday'].astype('category')
df['pickup_datetime_hour'] = df['pickup_datetime_hour'].astype('category')
df['pickup_datetime_month'] = df['pickup_datetime_month'].astype('category')
df['pickup_datetime_day'] = df['pickup_datetime_day'].astype('category')


In [ ]:
df

In [ ]:
test_df['pickup_datetime_year'] = test_df['pickup_datetime_year'].astype('category')
test_df['pickup_datetime_weekday'] = test_df['pickup_datetime_weekday'].astype('category')
test_df['pickup_datetime_hour'] = test_df['pickup_datetime_hour'].astype('category')
test_df['pickup_datetime_month'] = test_df['pickup_datetime_month'].astype('category')
test_df['pickup_datetime_day'] = test_df['pickup_datetime_day'].astype('category')

In [ ]:
test_df = test_df.drop(columns=['key'])

In [ ]:
test_df

In [ ]:
# testing some baseline models

In [ ]:
X = df.drop(columns=['fare_amount'])
y = df['fare_amount']

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error,root_mean_squared_error,r2_score

train_df,val_df,train_labels,val_labels = train_test_split(X.drop(columns=['pickup_datetime']),y,test_size=0.2,random_state=42)

In [ ]:
train_df

In [ ]:
from sklearn.linear_model import LinearRegression,Ridge

lr = LinearRegression()
lr.fit(train_df,train_labels)

In [ ]:
lr.predict(val_df)

In [ ]:
lr_pred = lr.predict(val_df)

In [ ]:
root_mean_squared_error(val_labels,lr_pred)

In [ ]:
lr.score(train_df,train_labels)

7.033 is rmse for basemodel like linear regreesion without any hyperparameter tuning



In [ ]:
''''
from sklearn.ensemble import RandomForestRegressor

RFR = RandomForestRegressor(n_estimators=100,
                            max_depth=3,random_state=42)
RFR.fit(train_df,train_labels)
'''

In [ ]:
# RFR_pred = RFR.predict(val_df)

In [ ]:
# root_mean_squared_error(val_labels,RFR_pred)

In [ ]:
'''6.073 is rmse for random forest regressor without any hyperparameter tuning'''

In [ ]:
'''
from sklearn.ensemble import GradientBoostingRegressor

GBR = GradientBoostingRegressor(n_estimators=100,
                                 max_depth=3,random_state=42,learning_rate = 0.1)
'''

In [ ]:
# GBR.fit(train_df,train_labels)

In [ ]:
# root_mean_squared_error(val_labels,GBR.predict(val_df))

3.974858 is a pretty good score without any hyperparameter tuning



In [ ]:
'''
from xgboost import XGBRegressor

XGB = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1,
    enable_categorical=True
)
'''

In [ ]:
# XGB.fit( train_df,train_labels)

In [ ]:
'''
XGB_pred = XGB.predict(val_df)
root_mean_squared_error(val_labels, XGB_pred)
'''

In [ ]:
# hyper-parameters tuning for Gradient-Boost and Xgboost algorithms


In [ ]:
'''
import matplotlib.pyplot as plt

def test_params(ModelClass, **params):
    """Trains a model with the given parameters and returns training & validation RMSE"""
    model = ModelClass(**params).fit(train_df, train_labels)
    train_rmse = root_mean_squared_error(model.predict(train_df), train_labels)
    val_rmse = root_mean_squared_error(model.predict(val_df), val_labels)
    return train_rmse, val_rmse

def test_param_and_plot(ModelClass, param_name, param_values, **other_params):
    """Trains multiple models by varying the value of param_name according to param_values"""
    train_errors, val_errors = [], []
    for value in param_values:
        params = dict(other_params)
        params[param_name] = value
        train_rmse, val_rmse = test_params(ModelClass, **params)
        train_errors.append(train_rmse)
        val_errors.append(val_rmse)

    plt.figure(figsize=(10,6))
    plt.title('Overfitting curve: ' + param_name)
    plt.plot(param_values, train_errors, 'b-o')
    plt.plot(param_values, val_errors, 'r-o')
    plt.xlabel(param_name)
    plt.ylabel('RMSE')
    plt.legend(['Training', 'Validation'])
    '''

In [ ]:
'''best_params = {
    'random_state': 42,
    'n_jobs': -1,
    'objective': 'reg:squarederror',
    'enable_categorical':True
}'''

In [ ]:
'''%%time
test_param_and_plot(XGBRegressor, 'n_estimators',[100,200,300,400,500], **best_params)'''

after n_estimator = 200 validation set has no impact on rmse but morethan 200 estimator cause overfitting in training dataset. so choosing 100-200 estimators are good in practice

In [ ]:
'''%%time
test_param_and_plot(XGBRegressor, 'learning_rate', [0.05,0.1,0.15,0.2,0.25,0.3,0.35,0.4,0.45,0.5], **best_params)'''

learning rate = 0.25

In [ ]:
'''%%time
test_param_and_plot(XGBRegressor, 'max_depth', [3,4,5,6,7], **best_params)'''

choosing max_depth = 5

In [ ]:
'''%%time
test_param_and_plot(XGBRegressor, 'subsample', [0.5,0.6,0.7,0.8,0.9], **best_params)'''

subsample has no significant impact on rmse after 0.8

In [ ]:
from xgboost import XGBRegressor

XGB_final = XGBRegressor(
    n_estimators=200,
    learning_rate=0.25,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1,
    enable_categorical=True)

XGB_final.fit(train_df,train_labels)

In [ ]:
XGB_final_pred = XGB_final.predict(val_df)
root_mean_squared_error(val_labels, XGB_final_pred)

In [ ]:
'''best_params = {
    'random_state': 42,
    'min_samples_leaf': 2,
    'criterion': 'squared_error',
    'min_samples_split': 2
}'''

In [ ]:
'''%%time
test_param_and_plot(GradientBoostingRegressor, 'learning_rate', [0.05,0.1,0.15,0.2,0.25,0.3,0.35,0.4,0.45,0.5], **best_params)'''

learning rate = 0.5

In [ ]:
'''%%time
test_param_and_plot(GradientBoostingRegressor, 'max_depth', [3,4,5,6,7], **best_params)'''

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

GBR_final = GradientBoostingRegressor(n_estimators=100,min_samples_leaf=2,criterion='squared_error',min_samples_split=2,
                                 max_depth=6,random_state=42,learning_rate = 0.5)
GBR_final.fit(train_df,train_labels)

In [ ]:
root_mean_squared_error(val_labels,GBR_final.predict(val_df))